In [3]:
"""
Fixed Feature Extraction for AI Crypto Trading Bot
Extracts comprehensive technical features from cryptocurrency data.
"""
import pandas as pd
import numpy as np

import os
import sys
import json

# Try to import talib with error handling
try:
    import talib
    TALIB_AVAILABLE = True
    print("✅ TA-Lib imported successfully")
except ImportError:
    print("❌ TA-Lib not available, installing alternative...")
    TALIB_AVAILABLE = False
    talib = None  # Set talib to None when not available
    # Install alternative technical analysis library
    try:
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "ta"])
        print("✅ Using 'ta' library as alternative")
    except Exception as e:
        print(f"❌ Failed to install alternative: {e}")
        ta = None
import logging
from datetime import datetime, timezone
from typing import Dict, List, Optional
import warnings

# Add src to path for imports
notebook_dir = os.getcwd()  # Get the current working directory in Jupyter Notebook
sys.path.append(os.path.join(notebook_dir, 'src'))  # Adjust the path to include the 'src' folder

warnings.filterwarnings('ignore')

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('feature_extraction_fixed.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# Configuration
DATA_FOLDER = 'data'
FEATURES_FOLDER = 'data/features'
BASE_SYMBOLS = ['BTC_USDT', 'ETH_USDT', 'SOL_USDT']
CONTEXT_SYMBOLS = ['ADA_USDT', 'MATIC_USDT', 'DOT_USDT', 'LINK_USDT', 'AVAX_USDT']
TIMEFRAMES = ['6h', '12h', '1d', '3d']

class FixedFeatureExtractor:
    """Fixed feature extraction for crypto trading"""
    
    def __init__(self):
        self.feature_stats = {}
        # Ensure folders exist
        os.makedirs(FEATURES_FOLDER, exist_ok=True)
    
    def load_data(self, symbol: str, timeframe: str) -> Optional[pd.DataFrame]:
        """Load OHLCV data for a symbol and timeframe"""
        try:
            filename = f"{symbol}_{timeframe}.csv"
            filepath = os.path.join(DATA_FOLDER, filename)
            
            if not os.path.exists(filepath):
                logger.warning(f"File not found: {filepath}")
                return None
            
            df = pd.read_csv(filepath, parse_dates=['timestamp'])
            df = df.sort_values('timestamp').reset_index(drop=True)
            
            if len(df) < 200:
                logger.warning(f"Insufficient data for {symbol} {timeframe}: {len(df)} candles")
                return None
            
            logger.info(f"Loaded {len(df)} candles for {symbol} {timeframe}")
            return df
            
        except Exception as e:
            logger.error(f"Error loading data for {symbol} {timeframe}: {e}")
            return None
    
    def extract_basic_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """Extract basic OHLCV-derived features"""
        features = df[['timestamp', 'open', 'high', 'low', 'close', 'volume']].copy()
        
        # Price-based features
        features['price_range'] = features['high'] - features['low']
        features['body_size'] = abs(features['close'] - features['open'])
        features['upper_shadow'] = features['high'] - np.maximum(features['open'], features['close'])
        features['lower_shadow'] = np.minimum(features['open'], features['close']) - features['low']
        
        # Relative features (avoid division by zero)
        features['range_pct'] = np.where(features['close'] > 0, 
                                        (features['price_range'] / features['close']) * 100, 0)
        features['body_pct'] = np.where(features['close'] > 0,
                                       (features['body_size'] / features['close']) * 100, 0)
        features['upper_shadow_pct'] = np.where(features['close'] > 0,
                                               (features['upper_shadow'] / features['close']) * 100, 0)
        features['lower_shadow_pct'] = np.where(features['close'] > 0,
                                               (features['lower_shadow'] / features['close']) * 100, 0)
        
        # Price position within range
        features['close_position'] = np.where(features['price_range'] > 0,
                                             (features['close'] - features['low']) / features['price_range'], 0.5)
        
        # Returns
        for period in [1, 3, 5, 10, 20]:
            features[f'return_{period}'] = features['close'].pct_change(period)
        
        # Log returns (handle zero/negative prices)
        features['log_return'] = np.where(
            (features['close'] > 0) & (features['close'].shift(1) > 0),
            np.log(features['close'] / features['close'].shift(1)),
            0
        )
        
        # Volume features
        features['volume_sma_10'] = features['volume'].rolling(10, min_periods=1).mean()
        features['volume_sma_20'] = features['volume'].rolling(20, min_periods=1).mean()
        features['volume_ratio'] = np.where(features['volume_sma_20'] > 0,
                                           features['volume'] / features['volume_sma_20'], 1)
        features['volume_return'] = features['volume'].pct_change(1)
        
        # Price-Volume relationship
        features['price_volume'] = features['close'] * features['volume']
        features['vwap_5'] = (features['price_volume'].rolling(5, min_periods=1).sum() / 
                             features['volume'].rolling(5, min_periods=1).sum())
        features['vwap_20'] = (features['price_volume'].rolling(20, min_periods=1).sum() / 
                              features['volume'].rolling(20, min_periods=1).sum())
        
        return features
        
    def extract_trend_indicators(self, df: pd.DataFrame) -> pd.DataFrame:
        """Extract trend-following indicators"""
        features = pd.DataFrame(index=df.index)
        
        if not TALIB_AVAILABLE or talib is None:
            logger.warning("TA-Lib not available, skipping trend indicators")
            return features
        
        try:
            # Moving Averages
            periods = [5, 10, 12, 20, 26, 50, 100, 200]
            close_vals = df['close'].values.astype(np.float64)
            for period in periods:
                if len(df) >= period:
                    features[f'sma_{period}'] = talib.SMA(close_vals, timeperiod=period)
                    features[f'ema_{period}'] = talib.EMA(close_vals, timeperiod=period)
                    
                    # Price relative to MA (avoid division by zero)
                    features[f'close_sma_{period}_ratio'] = np.where(
                        features[f'sma_{period}'] > 0,
                        df['close'] / features[f'sma_{period}'],
                        1
                    )
                    features[f'close_ema_{period}_ratio'] = np.where(
                        features[f'ema_{period}'] > 0,
                        df['close'] / features[f'ema_{period}'],
                        1
                    )
            
            # MA Crosses (only if we have enough data and the MAs exist)
            if len(df) >= 50:
                if 'sma_5' in features.columns and 'sma_20' in features.columns:
                    features['sma_5_20_cross'] = (features['sma_5'] > features['sma_20']).astype(int)
                if 'sma_10' in features.columns and 'sma_50' in features.columns:
                    features['sma_10_50_cross'] = (features['sma_10'] > features['sma_50']).astype(int)
                if 'ema_12' in features.columns and 'ema_26' in features.columns:
                    features['ema_12_26_cross'] = (features['ema_12'] > features['ema_26']).astype(int)
            
            # MACD
            if len(df) >= 34:  # Need at least 34 periods for MACD
                close_vals = df['close'].values.astype(np.float64)
                macd_line, macd_signal, macd_hist = talib.MACD(close_vals)
                features['macd'] = macd_line
                features['macd_signal'] = macd_signal
                features['macd_hist'] = macd_hist
                features['macd_cross'] = (macd_line > macd_signal).astype(int)
            
            # ADX and Directional Indicators
            if len(df) >= 14:
                high_vals = df['high'].values.astype(np.float64)
                low_vals = df['low'].values.astype(np.float64)
                close_vals = df['close'].values.astype(np.float64)
                
                features['adx'] = talib.ADX(high_vals, low_vals, close_vals, timeperiod=14)
            
            # Parabolic SAR
            if len(df) >= 2:
                high_vals = df['high'].values.astype(np.float64)
                low_vals = df['low'].values.astype(np.float64)
                features['sar'] = talib.SAR(high_vals, low_vals)
                features['sar_trend'] = (df['close'] > features['sar']).astype(int)
            
            # Aroon
            if len(df) >= 14:
                high_vals = df['high'].values.astype(np.float64)
                low_vals = df['low'].values.astype(np.float64)
                aroon_up, aroon_down = talib.AROON(high_vals, low_vals, timeperiod=14)
                features['aroon_up'] = aroon_up
                features['aroon_down'] = aroon_down
                features['aroon_osc'] = aroon_up - aroon_down
                
            # RSI variants
            close_vals = df['close'].values.astype(np.float64)
            if len(df) >= 14:
                features['rsi_14'] = talib.RSI(close_vals, timeperiod=14)
                features['rsi_oversold'] = (features['rsi_14'] < 30).astype(int)
                features['rsi_overbought'] = (features['rsi_14'] > 70).astype(int)
                features['rsi_neutral'] = ((features['rsi_14'] >= 40) & (features['rsi_14'] <= 60)).astype(int)
                
            if len(df) >= 7:
                features['rsi_7'] = talib.RSI(close_vals, timeperiod=7)
                
            if len(df) >= 21:
                features['rsi_21'] = talib.RSI(close_vals, timeperiod=21)
            
            # Stochastic
            if len(df) >= 14:
                high_vals = df['high'].values.astype(np.float64)
                low_vals = df['low'].values.astype(np.float64)
                close_vals = df['close'].values.astype(np.float64)
                stoch_k, stoch_d = talib.STOCH(high_vals, low_vals, close_vals)
                features['stoch_k'] = stoch_k
                features['stoch_d'] = stoch_d
                features['stoch_cross'] = (stoch_k > stoch_d).astype(int)
            
            # Williams %R
            if len(df) >= 14:
                high_vals = df['high'].values.astype(np.float64)
                low_vals = df['low'].values.astype(np.float64)
                close_vals = df['close'].values.astype(np.float64)
                features['willr'] = talib.WILLR(high_vals, low_vals, close_vals, timeperiod=14)
            
            # Rate of Change and Momentum
            close_vals = df['close'].values.astype(np.float64)
            for period in [5, 10, 20]:
                if len(df) >= period:
                    features[f'roc_{period}'] = talib.ROC(close_vals, timeperiod=period)
                    features[f'mom_{period}'] = talib.MOM(close_vals, timeperiod=period)
            
            # CCI
            if len(df) >= 14:
                high_vals = df['high'].values.astype(np.float64)
                low_vals = df['low'].values.astype(np.float64)
                close_vals = df['close'].values.astype(np.float64)
                features['cci'] = talib.CCI(high_vals, low_vals, close_vals, timeperiod=14)
            
            # Ultimate Oscillator
            if len(df) >= 28:  # Needs enough data
                high_vals = df['high'].values.astype(np.float64)
                low_vals = df['low'].values.astype(np.float64)
                close_vals = df['close'].values.astype(np.float64)
                features['ultosc'] = talib.ULTOSC(high_vals, low_vals, close_vals)
                
        except Exception as e:
            logger.warning(f"Error in trend indicators: {e}")
        
        return features
    def extract_volatility_indicators(self, df: pd.DataFrame) -> pd.DataFrame:
        """Extract volatility indicators"""
        features = pd.DataFrame(index=df.index)
        
        if not TALIB_AVAILABLE or talib is None:
            logger.warning("TA-Lib not available, skipping volatility indicators")
            return features
        
        try:
            # ATR
            if len(df) >= 14:
                high_vals = df['high'].values.astype(np.float64)
                low_vals = df['low'].values.astype(np.float64)
                close_vals = df['close'].values.astype(np.float64)
                atr = talib.ATR(high_vals, low_vals, close_vals, timeperiod=14)
                features['atr'] = atr
                features['atr_pct'] = np.where(df['close'] > 0, (atr / df['close']) * 100, 0)
            
            # Bollinger Bands
            if len(df) >= 20:
                close_vals = df['close'].values.astype(np.float64)
                bb_upper, bb_middle, bb_lower = talib.BBANDS(
                    close_vals, timeperiod=20, nbdevup=2, nbdevdn=2
                )
                features['bb_upper'] = bb_upper
                features['bb_middle'] = bb_middle
                features['bb_lower'] = bb_lower
                features['bb_width'] = bb_upper - bb_lower
                features['bb_position'] = np.where(
                    (bb_upper - bb_lower) > 0,
                    (df['close'] - bb_lower) / (bb_upper - bb_lower),
                    0.5
                )
                features['bb_squeeze'] = (features['bb_width'] < 
                                         features['bb_width'].rolling(20, min_periods=1).quantile(0.1)).astype(int)
            
            # Keltner Channels
            if len(df) >= 20:
                high_vals = df['high'].values.astype(np.float64)
                low_vals = df['low'].values.astype(np.float64)
                close_vals = df['close'].values.astype(np.float64)
                ema_20 = talib.EMA(close_vals, timeperiod=20)
                atr_10 = talib.ATR(high_vals, low_vals, close_vals, timeperiod=10)
                features['kc_upper'] = ema_20 + (2 * atr_10)
                features['kc_lower'] = ema_20 - (2 * atr_10)
                features['kc_position'] = np.where(
                    (features['kc_upper'] - features['kc_lower']) > 0,
                    (df['close'] - features['kc_lower']) / (features['kc_upper'] - features['kc_lower']),
                    0.5
                )
                
        except Exception as e:
            logger.warning(f"Error in volatility indicators: {e}")
        
        return features
    
    def extract_volume_indicators(self, df: pd.DataFrame) -> pd.DataFrame:
        """Extract volume indicators"""
        features = pd.DataFrame(index=df.index)
        
        if not TALIB_AVAILABLE or talib is None:
            logger.warning("TA-Lib not available, skipping volume indicators")
            return features
        
        try:
            for period in [5, 10, 20, 50]:
                if len(df) >= period:
                    vol_sma = df['volume'].rolling(period, min_periods=1).mean()
                    features[f'volume_sma_{period}'] = vol_sma
                    features[f'volume_ratio_{period}'] = np.where(vol_sma > 0, df['volume'] / vol_sma, 1)
            
            # Accumulation/Distribution Line
            if len(df) >= 1:
                high_vals = df['high'].values.astype(np.float64)
                low_vals = df['low'].values.astype(np.float64)
                close_vals = df['close'].values.astype(np.float64)
                volume_vals = df['volume'].values.astype(np.float64)
                features['ad'] = talib.AD(high_vals, low_vals, close_vals, volume_vals)
            
            # Money Flow Index
            if len(df) >= 14:
                high_vals = df['high'].values.astype(np.float64)
                low_vals = df['low'].values.astype(np.float64)
                close_vals = df['close'].values.astype(np.float64)
                volume_vals = df['volume'].values.astype(np.float64)
                features['mfi'] = talib.MFI(high_vals, low_vals, close_vals, volume_vals, timeperiod=14)
                
        except Exception as e:
            logger.warning(f"Error in volume indicators: {e}")
        
        return features
    
    def extract_candlestick_patterns(self, df: pd.DataFrame) -> pd.DataFrame:
        """Extract candlestick patterns"""
        features = pd.DataFrame(index=df.index)
        
        if not TALIB_AVAILABLE or talib is None:
            logger.warning("TA-Lib not available, skipping candlestick patterns")
            return features
        
        # Define pattern functions
        patterns = {
            'doji': talib.CDLDOJI,
            'hammer': talib.CDLHAMMER,
            'hanging_man': talib.CDLHANGINGMAN,
            'shooting_star': talib.CDLSHOOTINGSTAR,
            'engulfing': talib.CDLENGULFING,
            'harami': talib.CDLHARAMI,
            'morning_star': talib.CDLMORNINGSTAR,
            'evening_star': talib.CDLEVENINGSTAR,
            'three_black_crows': talib.CDL3BLACKCROWS,
            'three_white_soldiers': talib.CDL3WHITESOLDIERS
        }
        
        # Convert to numpy arrays once (initialize outside condition)
        open_vals = df['open'].values.astype(np.float64)
        high_vals = df['high'].values.astype(np.float64)
        low_vals = df['low'].values.astype(np.float64)
        close_vals = df['close'].values.astype(np.float64)
        
        # Extract patterns
        for name, func in patterns.items():
            try:
                if len(df) >= 3:  # Most patterns need at least 3 candles
                    pattern_result = func(open_vals, high_vals, low_vals, close_vals)
                    features[f'pattern_{name}'] = (pattern_result != 0).astype(int)
                    features[f'pattern_{name}_bullish'] = (pattern_result > 0).astype(int)
                    features[f'pattern_{name}_bearish'] = (pattern_result < 0).astype(int)
                else:
                    features[f'pattern_{name}'] = 0
                    features[f'pattern_{name}_bullish'] = 0
                    features[f'pattern_{name}_bearish'] = 0
            except Exception as e:
                logger.warning(f"Error extracting pattern {name}: {e}")
                features[f'pattern_{name}'] = 0
                features[f'pattern_{name}_bullish'] = 0
                features[f'pattern_{name}_bearish'] = 0
        
        # Pattern summary features
        try:
            bullish_cols = [col for col in features.columns if col.endswith('_bullish')]
            bearish_cols = [col for col in features.columns if col.endswith('_bearish')]
            
            if bullish_cols and bearish_cols:
                features['total_bullish_patterns'] = features[bullish_cols].sum(axis=1)
                features['total_bearish_patterns'] = features[bearish_cols].sum(axis=1)
                features['pattern_sentiment'] = features['total_bullish_patterns'] - features['total_bearish_patterns']
        except Exception as e:
            logger.warning(f"Error in pattern summary: {e}")
        
        return features
    
    def extract_market_structure_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """Extract market structure features"""
        features = pd.DataFrame(index=df.index)
        
        try:
            # Support and Resistance levels
            for window in [10, 20, 50]:
                if len(df) >= window:
                    resistance = df['high'].rolling(window, min_periods=1).max()
                    support = df['low'].rolling(window, min_periods=1).min()
                    
                    features[f'resistance_{window}'] = resistance
                    features[f'support_{window}'] = support
                    
                    # Distance to S/R levels
                    features[f'dist_to_resistance_{window}'] = np.where(
                        df['close'] > 0, (resistance - df['close']) / df['close'], 0
                    )
                    features[f'dist_to_support_{window}'] = np.where(
                        df['close'] > 0, (df['close'] - support) / df['close'], 0
                    )
            
            # Market structure breaks
            if len(df) >= 3:
                features['higher_high'] = (df['high'] > df['high'].shift(1)).astype(int)
                features['lower_low'] = (df['low'] < df['low'].shift(1)).astype(int)
                features['higher_low'] = ((df['low'] > df['low'].shift(1)) & 
                                         (df['low'].shift(1) < df['low'].shift(2))).astype(int)
                features['lower_high'] = ((df['high'] < df['high'].shift(1)) & 
                                         (df['high'].shift(1) > df['high'].shift(2))).astype(int)
            
            # Gap analysis
            if len(df) >= 2:
                features['gap_up'] = ((df['low'] > df['high'].shift(1)) & 
                                     (df['high'].shift(1) > 0)).astype(int)
                features['gap_down'] = ((df['high'] < df['low'].shift(1)) & 
                                       (df['low'].shift(1) > 0)).astype(int)
                
        except Exception as e:
            logger.warning(f"Error in market structure features: {e}")
        
        return features
    
    def add_timeframe_encoding(self, df: pd.DataFrame, timeframe: str) -> pd.DataFrame:
        """Add timeframe encoding features"""
        df = df.copy()

        # One-hot encode timeframes
        for tf in TIMEFRAMES:
            df[f'timeframe_{tf}'] = int(timeframe == tf)

        return df
    
    def clean_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """Clean and validate features"""
        try:
            initial_len = len(df)
            
            # Replace infinite values with NaN
            df = df.replace([np.inf, -np.inf], np.nan)
            
            # Fill NaN values with forward fill, then backward fill, then 0
            df = df.ffill().bfill().fillna(0)
            
            # Drop any remaining rows with NaN values
            df = df.dropna()
            
            final_len = len(df)
            
            if initial_len != final_len:
                logger.warning(f"Dropped {initial_len - final_len} rows due to NaN values")
            
            return df
            
        except Exception as e:
            logger.error(f"Error cleaning features: {e}")
            return df
    
    def extract_all_features(self, symbol: str, timeframe: str) -> Optional[pd.DataFrame]:
        """Extract all features for a symbol and timeframe"""
        try:
            # Load data
            df = self.load_data(symbol, timeframe)
            if df is None or df.empty:
                return None
            
            logger.info(f"Starting feature extraction for {symbol} {timeframe}...")
            
            all_features_list = []
            feature_counts = {}
            
            # Extract basic features (always available)
            logger.info("Extracting basic features...")
            try:
                basic_features = self.extract_basic_features(df)
                all_features_list.append(basic_features)
                feature_counts['basic'] = len(basic_features.columns)
            except Exception as e:
                logger.error(f"Failed to extract basic features: {e}")
                return None
            
            logger.info("Extracting trend indicators...")
            try:
                trend_features = self.extract_trend_indicators(df)
                if not trend_features.empty:
                    all_features_list.append(trend_features)
                    feature_counts['trend'] = len(trend_features.columns)
            except Exception as e:
                logger.error(f"Failed to extract trend features: {e}")
                feature_counts['trend'] = 0
            
            logger.info("Extracting volatility indicators...")
            try:
                volatility_features = self.extract_volatility_indicators(df)
                if not volatility_features.empty:
                    all_features_list.append(volatility_features)
                    feature_counts['volatility'] = len(volatility_features.columns)
            except Exception as e:
                logger.error(f"Failed to extract volatility features: {e}")
                feature_counts['volatility'] = 0
            
            logger.info("Extracting volume indicators...")
            try:
                volume_features = self.extract_volume_indicators(df)
                if not volume_features.empty:
                    all_features_list.append(volume_features)
                    feature_counts['volume'] = len(volume_features.columns)
            except Exception as e:
                logger.error(f"Failed to extract volume features: {e}")
                feature_counts['volume'] = 0
            
            logger.info("Extracting candlestick patterns...")
            try:
                pattern_features = self.extract_candlestick_patterns(df)
                if not pattern_features.empty:
                    all_features_list.append(pattern_features)
                    feature_counts['patterns'] = len(pattern_features.columns)
            except Exception as e:
                logger.error(f"Failed to extract pattern features: {e}")
                feature_counts['patterns'] = 0
            
            logger.info("Extracting market structure features...")
            try:
                structure_features = self.extract_market_structure_features(df)
                if not structure_features.empty:
                    all_features_list.append(structure_features)
                    feature_counts['structure'] = len(structure_features.columns)
            except Exception as e:
                logger.error(f"Failed to extract structure features: {e}")
                feature_counts['structure'] = 0
            
            if not all_features_list:
                logger.error(f"No features extracted for {symbol} {timeframe}")
                return None
            
            # Combine all features
            logger.info("Combining all features...")
            combined_features = pd.concat(all_features_list, axis=1)
            
            # Add timeframe encoding
            combined_features = self.add_timeframe_encoding(combined_features, timeframe)
            feature_counts['timeframe'] = len(TIMEFRAMES)
            
            # Clean features
            logger.info("Cleaning features...")
            combined_features = self.clean_features(combined_features)
            
            logger.info(f"✅ Extracted {len(combined_features.columns)} features for {len(combined_features)} candles")
            
            # Store stats
            self.feature_stats[f"{symbol}_{timeframe}"] = {
                'num_features': len(combined_features.columns),
                'num_candles': len(combined_features),
                'feature_groups': feature_counts
            }
            
            return combined_features
            
        except Exception as e:
            logger.error(f"Error extracting features for {symbol} {timeframe}: {e}")
            return None
    
    def save_features(self, features: pd.DataFrame, symbol: str, timeframe: str) -> str:
        """Save features to CSV file"""
        try:
            filename = f"features_{symbol}_{timeframe}.csv"
            filepath = os.path.join(FEATURES_FOLDER, filename)
            
            features.to_csv(filepath, index=False)
            
            logger.info(f"✅ Saved {len(features)} rows with {len(features.columns)} features to {filepath}")
            return filepath
            
        except Exception as e:
            logger.error(f"Failed to save features for {symbol} {timeframe}: {e}")
            return ""

def extract_features_for_all_symbols(symbols: List[str], timeframes: List[str]) -> Dict:
    """Extract features for all symbols and timeframes"""
    extractor = FixedFeatureExtractor()
    results = {
        'successful': [],
        'failed': [],
        'stats': {}
    }
    
    total_combinations = len(symbols) * len(timeframes)
    current_combination = 0
    
    logger.info(f"🚀 Starting feature extraction for {total_combinations} combinations")
    
    for symbol in symbols:
        for timeframe in timeframes:
            current_combination += 1
            progress = (current_combination / total_combinations) * 100
            
            logger.info(f"\nProgress: {progress:.1f}% ({current_combination}/{total_combinations})")
            
            try:
                # Extract features
                features = extractor.extract_all_features(symbol, timeframe)
                
                if features is not None and not features.empty:
                    # Save features
                    filepath = extractor.save_features(features, symbol, timeframe)
                    
                    if filepath:
                        results['successful'].append(f"{symbol}_{timeframe}")
                    else:
                        results['failed'].append(f"{symbol}_{timeframe}")
                else:
                    results['failed'].append(f"{symbol}_{timeframe}")
                    
            except Exception as e:
                logger.error(f"❌ Failed {symbol} {timeframe}: {e}")
                results['failed'].append(f"{symbol}_{timeframe}")
    
    results['stats'] = extractor.feature_stats
    return results

def create_feature_summary_report(results: Dict):
    """Create feature extraction summary report"""
    logger.info(f"\n📊 FEATURE EXTRACTION REPORT")
    logger.info(f"{'='*60}")
    
    total_features = 0
    total_candles = 0
    
    for key, stats in results['stats'].items():
        num_features = stats['num_features']
        num_candles = stats['num_candles']
        groups = stats['feature_groups']
        
        total_features += num_features
        total_candles += num_candles
        
        logger.info(f"{key}: {num_features} features, {num_candles} candles")
        logger.info(f"  Groups: {groups}")
    
    logger.info(f"\n📈 SUMMARY:")
    logger.info(f"Total feature sets: {len(results['stats'])}")
    if results['stats']:
        logger.info(f"Average features per set: {total_features / len(results['stats']):.0f}")
    logger.info(f"Total candles processed: {total_candles:,}")

def save_feature_summary(results: Dict):
    """Save feature extraction summary"""
    summary_report = {
        'extraction_timestamp': datetime.now(timezone.utc).isoformat(),
        'base_symbols': BASE_SYMBOLS,
        'context_symbols': CONTEXT_SYMBOLS,
        'timeframes': TIMEFRAMES,
        'successful_extractions': len(results['successful']),
        'failed_extractions': len(results['failed']),
        'successful_files': results['successful'],
        'failed_files': results['failed'],
        'detailed_stats': results['stats']
    }
    
    summary_path = os.path.join(FEATURES_FOLDER, 'feature_extraction_summary.json')
    with open(summary_path, 'w') as f:
        json.dump(summary_report, f, indent=2, default=str)
    
    logger.info(f"\n📋 Feature extraction summary saved to: {summary_path}")
    return summary_path

def main():
    """Main function to run feature extraction"""
    logger.info("🤖 AI CRYPTO TRADING BOT - FIXED FEATURE EXTRACTION")
    logger.info(f"🕐 {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')} UTC")
    logger.info(f"👤 User: samannazir55")
    logger.info("=" * 60)
    
    try:
        # Show configuration
        logger.info(f"Configuration:")
        logger.info(f"Base symbols: {BASE_SYMBOLS}")
        logger.info(f"Context symbols: {CONTEXT_SYMBOLS}")
        logger.info(f"Timeframes: {TIMEFRAMES}")
        logger.info(f"Data folder: {DATA_FOLDER}")
        logger.info(f"Features folder: {FEATURES_FOLDER}")
        
        # Extract features for all symbols
        all_symbols = BASE_SYMBOLS + CONTEXT_SYMBOLS
        results = extract_features_for_all_symbols(all_symbols, TIMEFRAMES)
        
        # Show results
        logger.info(f"\n{'='*60}")
        logger.info(f"📊 FEATURE EXTRACTION COMPLETE")
        logger.info(f"✅ Successful: {len(results['successful'])}")
        logger.info(f"❌ Failed: {len(results['failed'])}")
        
        if results['failed']:
            logger.info(f"\nFailed extractions:")
            for failed in results['failed']:
                logger.info(f"  - {failed}")
        
        # Create reports
        if results['successful']:
            create_feature_summary_report(results)
            summary_path = save_feature_summary(results)
            
            logger.info(f"\n✅ Feature extraction completed successfully!")
            logger.info(f"📋 Summary saved to: {summary_path}")
            logger.info(f"📁 Feature files saved to: {FEATURES_FOLDER}/")
            logger.info(f"\nNext step: Run 03_label_targets.py")
        else:
            logger.error(f"\n❌ No features were successfully extracted!")
            return False
        
        return True
        
    except Exception as e:
        logger.error(f"❌ Feature extraction failed: {e}")
        return False

if __name__ == "__main__":
    success = main()
    if not success:
        exit(1)

--- Logging error ---
Traceback (most recent call last):
  File "C:\ProgramData\anaconda3\Lib\logging\__init__.py", line 1154, in emit
    stream.write(msg + self.terminator)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\ProgramData\anaconda3\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\U0001f916' in position 33: character maps to <undefined>
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "d:\CryptoSight\venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "d:\CryptoSight\venv\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "d:\CryptoSight\venv\Lib\site-packages\ipykernel\kernelapp.py", line 739, in star

✅ TA-Lib imported successfully


--- Logging error ---
Traceback (most recent call last):
  File "C:\ProgramData\anaconda3\Lib\logging\__init__.py", line 1154, in emit
    stream.write(msg + self.terminator)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\ProgramData\anaconda3\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\u2705' in position 33: character maps to <undefined>
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "d:\CryptoSight\venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "d:\CryptoSight\venv\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "d:\CryptoSight\venv\Lib\site-packages\ipykernel\kernelapp.py", line 739, in start
  